# Exercise 1.1 – Detecting and Tracking Moving Cars (Traffic_Laramie_1.mp4)

## Objective
This notebook implements a motion-based computer vision application to **detect and track moving cars** from a traffic camera recording (`Traffic_Laramie_1.mp4`).

The implementation is written in **Python** using **OpenCV** and is based on the two required techniques:
- **Frame differencing**
- **Background subtraction**

To reduce false detections and improve performance, processing is restricted to the **main street Region of Interest (ROI)**.

In [ ]:
!pip -q install opencv-python numpy matplotlib

## Imports
We import OpenCV for image/video processing and NumPy for numerical operations.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
try:
    from google.colab.patches import cv2_imshow
    def show_image(img):
        cv2_imshow(img)
except ImportError:
    def show_image(img):
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 6))
        plt.imshow(img_rgb)
        plt.axis("off")
        plt.show()

## Load the video
The input is the client-provided traffic camera recording: `Traffic_Laramie_1.mp4`.

In [ ]:
video_path = "Traffic_Laramie_1.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise FileNotFoundError("Could not open video. Check that Traffic_Laramie_1.mp4 is uploaded.")

## Region of Interest (ROI) – Main Street
To reduce noise from buildings, pedestrians, and parked cars, we restrict processing to the **main street** region.

The ROI used here is the lower part of the frame:
- x = full width
- y = bottom ~45% of the frame

This improves detection stability and reduces false positives.

In [ ]:
ret, frame0 = cap.read()
if not ret:
    raise RuntimeError("Could not read the first frame from the video.")

h, w = frame0.shape[:2]
print("Frame size (h, w):", (h, w))

# ROI for main street (tuned based on your preview)
x1, x2 = 0, w
y1, y2 = int(h * 0.55), h

roi_preview = frame0[y1:y2, x1:x2].copy()
show_image(roi_preview)

## Frame Differencing (Motion Detection)
Frame differencing detects motion by measuring changes between consecutive frames.

Steps:
1. Convert ROI to grayscale.
2. Apply Gaussian blur to reduce noise.
3. Compute absolute difference between current and previous frame.
4. Threshold the difference image to obtain a binary motion mask.

This highlights pixels that changed significantly, indicating moving objects.

### Background Subtraction (MOG2)

Background subtraction models the scene background over time and identifies pixels that deviate from this model as foreground (moving objects).

In this implementation, OpenCV’s **BackgroundSubtractorMOG2** algorithm is used to detect moving vehicles in the video.

This method has several advantages:
- It **adapts to gradual lighting changes**, which commonly occur in outdoor traffic scenes.
- It models the background using a **Gaussian mixture model**, allowing it to distinguish between stable background pixels and moving foreground objects.
- It can **detect and label shadows**, which helps improve motion segmentation.

The background subtractor is created with the following parameters:

- **history = 500**  
  The model uses the last ~500 frames to estimate the background. A longer history improves stability but requires a short warm-up period.

- **varThreshold = 50**  
  Controls how sensitive the algorithm is to changes. This value helps detect vehicles while reducing false positives caused by small lighting variations.

- **detectShadows = True**  
  Enables shadow detection. Since shadows can appear as false foreground regions, they are removed in a later step using thresholding.

Compared to frame differencing alone, MOG2 typically produces a **cleaner and more stable foreground mask**, which improves vehicle detection reliability.

## Combining Motion Masks + Morphological Processing
To improve robustness, we combine:
- frame differencing mask
- background subtraction mask

using a bitwise OR operation.

Then we apply morphological operations:
- **Opening**: removes small noise blobs.
- **Dilation**: fills gaps and enlarges detected regions for full object coverage.

The final result is a cleaner motion mask for car detection.



## Car Detection and Tracking
### Detection
Cars are detected by:
1. Finding contours in the final motion mask.
2. Filtering out small contours using an area threshold.
3. Drawing bounding boxes around remaining contours.

### Tracking
Tracking is centroid-based:
- Each detection has a centroid.
- Detections are matched to existing tracks using nearest-neighbour distance.
- New detections create new IDs.
- Tracks are removed if missing for several frames.

This maintains consistent IDs across frames (basic multi-object tracking).

### Key Parameters (and why)
- `diff threshold = 15`: removes tiny pixel changes caused by noise; keeps real vehicle motion.
- `fg threshold = 200`: removes MOG2 shadow labels (shadows are lower intensity than true foreground).
- `kernel (5×5) + opening + dilation`: opening removes small blobs; dilation fills gaps so the full car shape is captured.
- `area threshold = 900`: filters out pedestrians/small movements so we focus on vehicles in the main street ROI.
- `max_match_dist = 60`: a detection is matched to an existing ID only if the centroid is close enough (prevents wrong ID swaps).
- `max_missed_frames = 20`: keeps an ID alive briefly during occlusion (e.g., car passes behind another).

In [ ]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

bg_sub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=True)
prev_gray = None

next_id = 1
tracks = {}      # id -> (cx, cy)
missed = {}      # id -> missed frames

max_match_dist = 60
max_missed_frames = 20

def box_centroid(x, y, w_box, h_box):
    return (int(x + w_box/2), int(y + h_box/2))

frame_idx = 0
skip = 60

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1
    roi = frame[y1:y2, x1:x2]

    fg = bg_sub.apply(roi)
    _, fg_mask = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    if prev_gray is None:
        prev_gray = gray
        continue

    diff = cv2.absdiff(gray, prev_gray)
    _, diff_mask = cv2.threshold(diff, 15, 255, cv2.THRESH_BINARY)
    prev_gray = gray

    if frame_idx < skip:
        continue

    motion_mask = cv2.bitwise_or(fg_mask, diff_mask)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    motion_mask = cv2.dilate(motion_mask, kernel, iterations=2)

    contours, _ = cv2.findContours(motion_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    detections = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < 900:
            continue
        x, y, w_box, h_box = cv2.boundingRect(c)
        cx, cy = box_centroid(x, y, w_box, h_box)
        detections.append({
            "box": (x, y, w_box, h_box),
            "c": (cx, cy),
            "id": None
        })

    # increment missed counters
    for tid in list(missed.keys()):
        missed[tid] += 1

    # build matching pairs
    pairs = []
    for di, det in enumerate(detections):
        cx, cy = det["c"]
        for tid, (tx, ty) in tracks.items():
            d = ((cx - tx)**2 + (cy - ty)**2) ** 0.5
            if d <= max_match_dist:
                pairs.append((d, tid, di))

    pairs.sort(key=lambda x: x[0])
    used_tracks = set()
    used_dets = set()

    for d, tid, di in pairs:
        if tid in used_tracks or di in used_dets:
            continue

        used_tracks.add(tid)
        used_dets.add(di)

        detections[di]["id"] = tid
        tracks[tid] = detections[di]["c"]
        missed[tid] = 0

    # new detections -> new IDs
    for det in detections:
        if det["id"] is None:
            det["id"] = next_id
            tracks[next_id] = det["c"]
            missed[next_id] = 0
            next_id += 1

    # remove old tracks
    for tid in list(missed.keys()):
        if missed[tid] > max_missed_frames:
            missed.pop(tid, None)
            tracks.pop(tid, None)

    # show every 10 frames
    if frame_idx % 10 == 0:

        # draw on ROI first
        vis_roi = roi.copy()

        for det in detections:
            x, y, w_box, h_box = det["box"]
            cx, cy = det["c"]
            tid = det["id"]

            cv2.rectangle(vis_roi, (x, y), (x + w_box, y + h_box), (0, 255, 0), 2)
            cv2.putText(vis_roi, f"ID {tid}", (x, max(20, y - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            cv2.circle(vis_roi, (cx, cy), 3, (255, 255, 0), -1)

        # overlay ROI back into full frame
        vis_full = frame.copy()
        vis_full[y1:y2, x1:x2] = vis_roi

        print(f"Frame {frame_idx} | Detections {len(detections)} | Active tracks {len(tracks)}")
        show_image(vis_full)

cap.release()


## Results and Observations
The application:
- Detects moving vehicles within the main street ROI using motion segmentation.
- Draws bounding boxes around detected cars.
- Assigns and maintains track IDs across frames using centroid matching.

### Notes
- Thresholds (motion threshold, contour area threshold) may be adjusted to improve robustness.
- Occasional ID switching can occur in basic centroid trackers when vehicles overlap or leave/re-enter the ROI.

##Further Development Implemented

- Region of Interest (ROI) restriction: Processing is limited to the “main street” area (lower portion of the frame) to reduce false detections from buildings/sidewalk/parked cars and improve speed.

- Noise reduction & mask cleanup (morphology): After combining motion masks, morphological opening removes small blobs and dilation fills gaps so car regions become more complete before contour detection.

- Shadow removal in background subtraction: MOG2 can label shadows; applying a high threshold (e.g., 200) keeps mainly true foreground motion and suppresses shadows.

- Tracking stability controls: A maximum match distance is used for nearest-neighbour association and tracks are removed after a number of missed frames, reducing ID drift and preventing “dead” tracks from staying forever.

- Warm-up/skip frames: Initial frames are skipped to allow the background model to stabilise before detection begins, improving robustness.

##Future Improvements
Potential enhancements include:
- **Kalman filtering** to smooth motion estimates and reduce ID switching.
- **Hungarian assignment** for more optimal multi-object matching.
- Deep learning detectors (e.g., **YOLO**) for more accurate car detection without relying on motion only.
- Improved shadow removal and adaptive thresholds for changing lighting conditions.